# HunyuanVideo-1.5 on a free Kaggle T4 — benchmark / selftest (no tunnel, no server)

**What it does:** installs diffusers + bitsandbytes, loads HunyuanVideo-1.5 **480p image-to-video step-distilled** in 4-bit (NF4),
animates one test image into a short vertical clip, and prints the timings. Output: `/kaggle/working/hy15_test.mp4` + `hy15_bench.json`.

**Why this exists:** the real pipeline uses the YouTube Auto **GPU Router/Bridge** kernel (private batch, pulls jobs from the Drive queue).
Run this notebook **once by hand** to learn if HunyuanVideo is fast enough on a T4 before switching `MPT_CLIP_MODEL=hy15-i2v`.
Never run it while the Router's kernel is running (one Kaggle account, one GPU session at a time, 30 h/week shared).

**Settings:** Accelerator = *GPU T4 x2* (only GPU 0 is used), Internet = On, phone-verified account.
Expect ~10–20 min for the ~35 GB download + load, then the timed generation. Stop the session when it finishes.

In [ ]:
import os, subprocess, sys, time, json
os.environ["HF_HOME"] = "/tmp/hf"            # ~35 GB of weights: keep out of /kaggle/working (20 GB limit)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "diffusers>=0.36", "transformers", "accelerate",
                "bitsandbytes>=0.45", "sentencepiece", "imageio", "imageio-ffmpeg", "ftfy", "protobuf"], check=True)
subprocess.run(["nvidia-smi"]); subprocess.run(["df", "-h", "/tmp", "/kaggle/working"]); subprocess.run(["free", "-g"])

In [ ]:
# Settings you may change
STEPS = 8            # step-distilled i2v: 8 = fast, 12 = recommended quality, 4 = fastest
FRAMES = 61          # must be 4k+1. 61 @24fps ~2.5 s, 121 = 5 s (much slower on a T4)
DTYPE = "fp16"       # T4 has no native bf16. If the clip is black/NaN, set "bf16" and re-run
PROMPT = ("A stack of gold coins on a dark wooden desk slowly grows as new coins drop in, warm rim light, "
          "shallow depth of field, slow dolly in, cinematic photoreal, vertical 9:16 framing")
REPO = "hunyuanvideo-community/HunyuanVideo-1.5-Diffusers-480p_i2v_step_distilled"

In [ ]:
import torch, numpy as np
from PIL import Image
from diffusers import HunyuanVideo15ImageToVideoPipeline, PipelineQuantizationConfig
from diffusers.utils import export_to_video
dt = torch.bfloat16 if DTYPE == "bf16" else torch.float16
t0 = time.time()
q = PipelineQuantizationConfig(quant_backend="bitsandbytes_4bit",
        quant_kwargs={"load_in_4bit": True, "bnb_4bit_quant_type": "nf4", "bnb_4bit_compute_dtype": dt},
        components_to_quantize=["transformer", "text_encoder"])
pipe = HunyuanVideo15ImageToVideoPipeline.from_pretrained(REPO, quantization_config=q, torch_dtype=dt)
pipe.enable_model_cpu_offload(); pipe.vae.enable_tiling()
t_load = time.time() - t0; print(f"download+load {t_load/60:.1f} min")

In [ ]:
# Test first frame: a warm 480x832 gradient (in production this is MoneyPrinter's FLUX scene image)
y = np.linspace(0, 1, 832)[:, None, None]; x = np.linspace(0, 1, 480)[None, :, None]
img = Image.fromarray((np.concatenate([0.9-0.4*y+0*x, 0.6-0.3*y+0.1*x, 0.2+0.2*x+0*y], 2)*255).astype("uint8"))
torch.cuda.reset_peak_memory_stats(); t1 = time.time()
frames = pipe(image=img, prompt=PROMPT, num_frames=FRAMES, num_inference_steps=STEPS,
              generator=torch.Generator("cpu").manual_seed(42)).frames[0]
t_gen = time.time() - t1
mid = np.asarray(frames[len(frames)//2], dtype=np.float32)
ok = bool(np.isfinite(mid).all() and mid.max() > 0.02)
export_to_video(frames, "/kaggle/working/hy15_test.mp4", fps=24)
info = {"steps": STEPS, "frames": FRAMES, "dtype": DTYPE, "load_min": round(t_load/60, 1), "gen_min": round(t_gen/60, 2),
        "peak_vram_gb": round(torch.cuda.max_memory_allocated()/1e9, 1), "frames_ok": ok,
        "clips_per_week_1gpu_est": int((29 * 60) / max(t_gen/60 + 0.3, 0.1))}
json.dump(info, open("/kaggle/working/hy15_bench.json", "w")); print(info)
if not ok: print("Black/NaN output: set DTYPE='bf16' and run again")

**Read the result:** `gen_min` is the time for one clip. Send `hy15_bench.json` to MoneyPrinter / Grok Bot.
Rule of thumb: if `gen_min` ≤ 8, HunyuanVideo i2v is worth using for MPT Shorts (≈200+ clips/week on one T4);
if it is > 15, stay on Wan2.1-1.3B (both T4s) or LTX-Video. **Stop the session now** (Run → Stop session) to save quota.